# Template Subtraction - Keep Only the Student's Marks

Approach 2, step 1. Every student fills in the same printed worksheet. The only difference between two copies of the same exercise is what the student wrote, ticked or coloured. This notebook removes the printed worksheet so that only the student's marks remain.

**Steps, for each exercise**:
1. Align all the copies on a reference copy (same position, angle, scale, perspective)
2. Normalise the paper brightness (scans and phone photos differ)
3. Rebuild the blank worksheet from the copies themselves (no blank scan is available)
4. Subtract it: what is darker than the blank worksheet is a student mark

**Input**: `data/labeled/by_exercise/qXX/correct|incorrect/*.jpg` (output of `01_cleaning.ipynb` and of the manual sorting)

**Outputs** (not versioned):
- `data/templates/qXX.png`: the rebuilt blank worksheet of each exercise
- `data/aligned/qXX/<copy>.png`: each copy aligned on the template (same coordinates for every copy of an exercise)
- `data/marks/qXX/<copy>.png`: the student's marks only (black on white)

All the images of an exercise share the same coordinates, so an answer zone defined once on the template applies to every copy.

## 1. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

LABELED_DIR = Path("../../data/labeled/by_exercise")
TEMPLATES_DIR = Path("../../data/templates")
ALIGNED_DIR = Path("../../data/aligned")
MARKS_DIR = Path("../../data/marks")

WORK_WIDTH = 1200          # every copy is resized to this width before alignment
MAX_MATCHES = 800          # keypoint matches kept for the alignment
MIN_INLIERS = 60           # below this, the alignment is considered failed
# Blank worksheet settings, per batch (see section 4)
#   percentile: brightness percentile of the copies used to rebuild the blank worksheet
#   line_margin: thickening of the printed lines, absorbs small alignment errors (pixels)
TEMPLATE_SETTINGS = {
    1: {"percentile": 100, "line_margin": 13},   # batch 1: flat scans, well aligned, only 5 copies
    2: {"percentile": 60, "line_margin": 7},     # batch 2: phone photos, slightly curved, 10 copies
}
MARK_THRESHOLD = 25        # darker than the template by more than this = student mark (0-255)

for folder in (TEMPLATES_DIR, ALIGNED_DIR, MARKS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

## 2. Loading the copies of an exercise

Each copy is converted to grayscale and resized to the same width. The batch 1 scans have two resolutions (1519 and 2281 pixels wide), this puts them at the same scale.

In [ ]:
def load_exercise(exercise):
    """Returns the paths and grayscale images (same width) of all the copies of an exercise."""
    paths = sorted(p for p in (LABELED_DIR / exercise).glob("*/*") if p.suffix.lower() in (".jpg", ".png", ".jpeg"))
    images = []
    for path in paths:
        gray = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)
        height = int(WORK_WIDTH * gray.shape[0] / gray.shape[1])
        images.append(cv2.resize(gray, (WORK_WIDTH, height), interpolation=cv2.INTER_AREA))
    return paths, images


exercises = sorted(d.name for d in LABELED_DIR.iterdir() if d.is_dir())
print(f"{len(exercises)} exercises: {', '.join(exercises)}")

## 3. Alignment

The printed worksheet gives many reference points (corners, outlines of the drawings). We detect them with ORB, match them between each copy and a reference copy, and compute the homography that maps the copy onto the reference (RANSAC ignores the wrong matches).

The reference is the copy that matches best with all the others. A copy with fewer than `MIN_INLIERS` consistent matches is reported: its alignment is not reliable.

In [ ]:
orb = cv2.ORB_create(5000)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)


def align_exercise(images):
    """Aligns every image on the best reference image. Returns the aligned images and the number of inliers."""
    features = [orb.detectAndCompute(img, None) for img in images]

    def total_matches(i):
        return sum(len(matcher.match(features[i][1], features[j][1])) for j in range(len(images)) if j != i)

    ref = max(range(len(images)), key=total_matches)
    ref_keypoints, ref_descriptors = features[ref]
    height, width = images[ref].shape

    aligned, inliers = [], []
    for i, img in enumerate(images):
        if i == ref:
            aligned.append(img)
            inliers.append(MAX_MATCHES)
            continue
        keypoints, descriptors = features[i]
        matches = sorted(matcher.match(descriptors, ref_descriptors), key=lambda m: m.distance)[:MAX_MATCHES]
        src = np.float32([keypoints[m.queryIdx].pt for m in matches])
        dst = np.float32([ref_keypoints[m.trainIdx].pt for m in matches])
        homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, 4.0)
        n_inliers = int(mask.sum()) if homography is not None else 0
        if n_inliers >= MIN_INLIERS:
            aligned.append(cv2.warpPerspective(img, homography, (width, height), borderValue=255))
        else:
            aligned.append(cv2.resize(img, (width, height)))  # fallback: simple resize
        inliers.append(n_inliers)
    return aligned, inliers

## 4. Paper normalisation and blank worksheet

**Paper normalisation**: the paper is not equally white everywhere (lighting of the phone photos, shadows). We estimate the paper colour around each pixel (a large maximum filter removes the ink, then a blur) and divide by it, so that the paper becomes uniformly white.

**Blank worksheet**: no blank scan is available, so we rebuild it from the copies. A printed pixel is dark on every copy, a student mark is dark only on some copies. For each pixel we take a bright percentile of the copies: a pixel stays in the template only if it is dark on enough copies. The printed lines are then thickened, so that a small alignment error does not leave them in the marks.

The settings depend on the batch (`TEMPLATE_SETTINGS`), because the two batches do not have the same weak point:
- **Batch 1** (flat scans, only 5 copies per exercise): the alignment is precise, but the same correct answer is often written at the same place by 3 or 4 students out of 5. With the median, this answer would stay in the template and disappear from their marks (seen on q19 and q24). We take the brightest copy (percentile 100): a pixel is in the template only if it is dark on every copy. The lines are thickened a lot (13 px) to absorb the small shifts.
- **Batch 2** (phone photos, 10 copies): the paper is slightly curved and the alignment less precise. With the brightest copy, printed lines that are slightly misaligned would disappear from the template and come back in the marks. A percentile of 60 keeps them, and still protects an answer given by up to about 40 % of the students.

These values were chosen by comparing the marks obtained with several settings (percentile 50, 60, 75, 100 and thickening 5 to 13 px).

In [ ]:
def normalise_paper(img):
    """Divides the image by the paper colour around each pixel, so that the paper becomes white."""
    paper = cv2.dilate(img, np.ones((25, 25), np.uint8))   # removes the ink, keeps the paper
    paper = cv2.GaussianBlur(paper, (0, 0), 20)
    return np.clip(img.astype(np.float32) / np.maximum(paper, 1) * 255, 0, 255).astype(np.uint8)


def batch_of(path):
    """Batch of a copy, from its file name: "b2_set_4_q33" is batch 2, "set_2_q20" is batch 1."""
    return 2 if path.stem.startswith("b2_") else 1


def build_template(images, batch):
    """Blank worksheet: bright percentile of the copies, printed lines thickened."""
    settings = TEMPLATE_SETTINGS[batch]
    template = np.percentile(np.stack(images), settings["percentile"], axis=0).astype(np.uint8)
    margin = settings["line_margin"]
    return cv2.erode(template, np.ones((margin, margin), np.uint8))  # erode on white = thicker dark lines

## 5. Subtraction

What is darker than the blank worksheet by more than `MARK_THRESHOLD` is a student mark. A small morphological opening removes the isolated noise pixels of the scans. The result is black marks on a white background.

In [ ]:
def extract_marks(img, template):
    """Student marks only: black on white."""
    darker = np.clip(template.astype(np.int16) - img, 0, 255).astype(np.uint8)
    darker = cv2.GaussianBlur(darker, (5, 5), 0)
    marks = (darker > MARK_THRESHOLD).astype(np.uint8) * 255
    marks = cv2.morphologyEx(marks, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    return 255 - marks

## 6. Processing all the exercises

For each exercise: alignment, normalisation, template, subtraction, then saving. The copies keep their file name (`set_2_q20`, `b2_set_4_q33`), the label is not repeated: it stays in `data/labeled/`.

In [ ]:
report = []
for exercise in exercises:
    paths, images = load_exercise(exercise)
    aligned, inliers = align_exercise(images)
    aligned = [normalise_paper(img) for img in aligned]
    template = build_template(aligned, batch_of(paths[0]))  # all the copies of an exercise are in the same batch

    (ALIGNED_DIR / exercise).mkdir(exist_ok=True)
    (MARKS_DIR / exercise).mkdir(exist_ok=True)
    cv2.imwrite(str(TEMPLATES_DIR / f"{exercise}.png"), template)
    for path, img, n_inliers in zip(paths, aligned, inliers):
        marks = extract_marks(img, template)
        cv2.imwrite(str(ALIGNED_DIR / exercise / f"{path.stem}.png"), img)
        cv2.imwrite(str(MARKS_DIR / exercise / f"{path.stem}.png"), marks)
        report.append({"exercise": exercise, "copy": path.stem, "label": path.parent.name,
                       "inliers": n_inliers, "aligned": n_inliers >= MIN_INLIERS,
                       "marks_pixels_%": round((marks < 128).mean() * 100, 2)})

report = pd.DataFrame(report)
print(f"{len(report)} copies processed, {(~report['aligned']).sum()} alignment failures")
display(report.sort_values("inliers").head(5))

## 7. Visual check

For a few exercises, the template, then each copy (aligned) and its marks. The marks must contain the student's answer and almost nothing of the printed worksheet.

In [ ]:
def show_exercise(exercise, n_copies=5):
    copies = report[report["exercise"] == exercise].head(n_copies)
    fig, axes = plt.subplots(2, n_copies + 1, figsize=(3.2 * (n_copies + 1), 6))
    template = cv2.imread(str(TEMPLATES_DIR / f"{exercise}.png"), cv2.IMREAD_GRAYSCALE)
    axes[0, 0].imshow(template, cmap="gray")
    axes[0, 0].set_title(f"{exercise} template")
    axes[1, 0].axis("off")
    for k, row in enumerate(copies.itertuples(), start=1):
        aligned = cv2.imread(str(ALIGNED_DIR / exercise / f"{row.copy}.png"), cv2.IMREAD_GRAYSCALE)
        marks = cv2.imread(str(MARKS_DIR / exercise / f"{row.copy}.png"), cv2.IMREAD_GRAYSCALE)
        axes[0, k].imshow(aligned, cmap="gray")
        axes[0, k].set_title(f"{row.copy} ({row.label})", fontsize=9)
        axes[1, k].imshow(marks, cmap="gray")
        axes[1, k].set_title("marks", fontsize=9)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()


for exercise in ["q12", "q24", "q01", "q33"]:
    show_exercise(exercise)

## 8. Known limits

- **Phone photos (batch 2)**: the paper is slightly curved, a single homography cannot align the whole page perfectly. Some printed outlines remain in the marks, mostly far from the centre. They are removed later by keeping only the answer zones.
- **Very pale pencil**: a stroke lighter than `MARK_THRESHOLD` is lost. Lowering the threshold keeps more strokes but also more scan noise.
- **Rebuilt template, batch 1**: an answer is lost only if every student draws exactly the same mark at the same place. The strong thickening (13 px) can also erase a stroke that touches a printed line.
- **Rebuilt template, batch 2**: an answer given at the same place by more than about 40 % of the students can stay in the template.
- **Printed box borders**: some straight lines of the answer boxes remain in the marks. They are removed when the answer zones are cut out.